In [2]:
!pip install rapidfuzz
!pip install re

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.7 MB 1.1 MB/s eta 0:00:02
   ------ --------------------------------- 0.3/1.7 MB 2.3 MB/s eta 0:00:01
   -------------- ------------------------- 0.6/1.7 MB 4.0 MB/s eta 0:00:01
   ----------------------- ---------------- 1.0/1.7 MB 5.0 MB/s eta 0:00:01
   ----------------------------------- ---- 1.5/1.7 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------  1.7/1.7 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 5.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import numpy as np, pandas as pd
from rapidfuzz import fuzz, process

# ---------- 1. Load ----------
h = pd.read_csv("C:\\CareNavigator\\ml\\data\\hospitals.csv", dtype=str)
p = pd.read_excel("C:\\CareNavigator\\ml\\data\\PMJAY_hospital_data.xls", dtype=str)      # add header=N if needed

def snake(c): return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
h.columns = [snake(c) for c in h.columns]
p.columns = [snake(c) for c in p.columns]
h = h.replace({"0": pd.NA}).copy()
p = p.dropna(how="all")

# ---------- 2. Telangana filters ----------
h["latitude"]  = pd.to_numeric(h["latitude"],  errors="coerce")
h["longitude"] = pd.to_numeric(h["longitude"], errors="coerce")
pin = pd.to_numeric(h["pincode"], errors="coerce")

is_tg = (
    h["state"].str.contains("telangana", case=False, na=False)
    | pin.between(500000, 509999)
)
# drop rows whose coordinates are clearly outside Telangana (rough bounding box)
bad_geo = h.latitude.notna() & ~(h.latitude.between(15.8, 19.95) & h.longitude.between(77.2, 81.8))
h = h[is_tg & ~bad_geo].reset_index(drop=True)

p = p[p["state"].str.contains("telangana", case=False, na=False)].reset_index(drop=True)
print("hospitals:", len(h), "| PMJAY rows:", len(p))
print(p["application_status"].value_counts(dropna=False))

hospitals: 3468 | PMJAY rows: 1733
application_status
Hospital Approved for Empanelment    1733
Name: count, dtype: int64


In [3]:
ACTIVE = "approved|empanel|active"          # adjust to the real status values you see
p = p[p["application_status"].str.contains(ACTIVE, case=False, na=False)].reset_index(drop=True)

In [6]:
# ---------- 3. Normalisation ----------
STOP = {"hospital","hospitals","pvt","private","ltd","limited","and","the","multispeciality",
        "multi","speciality","super","clinic","centre","center","nursing","home","medical",
        "research","institute","health","care"}

def norm_name(s):
    s = re.sub(r"[^a-z0-9 ]+", " ", str(s).lower())
    toks = [t for t in s.split() if t not in STOP]
    return " ".join(toks)

DIST_ALIAS = {"rangareddy": "rangareddy", "rangareddi": "rangareddy",
              "medchalmalkajgiri": "medchal", "medchal": "medchal",
              "hyderabad": "hyderabad", "sangareddy": "sangareddy", "medak": "medak"}
def norm_dist(s):
    d = re.sub(r"[^a-z]+", "", str(s).lower())
    d = re.sub(r"(urban|rural)$", "", d) or d
    return DIST_ALIAS.get(d, d)

def norm_phone(s):
    d = re.sub(r"\D", "", str(s))
    return d[-8:] if len(d) >= 8 else None   # last 8 digits works for landline and mobile

for df, phone_col in ((h, "phone"), (p, "hospital_contact")):
    df["n_name"]  = df["hospital_name"].map(norm_name)
    df["n_dist"]  = df["district"].map(norm_dist)
    df["n_phone"] = df[phone_col].map(norm_phone) if phone_col in df else None

In [25]:
import numpy as np
import pandas as pd
from collections import defaultdict
from rapidfuzz import fuzz, process

# ---------- 4. Fuzzy match ----------

# --------------------------------------------------
# 0. Normalize names
# --------------------------------------------------

pn = (
    p["n_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
    .tolist()
)

hn = (
    h["n_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
    .tolist()
)


# --------------------------------------------------
# 1. Name similarity
# --------------------------------------------------

S = 0.5 * (
    process.cdist(
        pn,
        hn,
        scorer=fuzz.token_sort_ratio,
        workers=-1,
        dtype=np.float32
    )
    +
    process.cdist(
        pn,
        hn,
        scorer=fuzz.token_set_ratio,
        workers=-1,
        dtype=np.float32
    )
)


# --------------------------------------------------
# 2. District bonus
# --------------------------------------------------

areas = (
    pd.concat(
        [p["n_dist"], h["n_dist"]],
        ignore_index=True
    )
    .replace("", np.nan)
)

codes, _ = pd.factorize(areas)

pc = codes[:len(p)]
hc = codes[len(p):]

same_dist = (
    (pc[:, None] == hc[None, :])
    & (pc[:, None] >= 0)
)

S += np.where(
    same_dist,
    6,
    0
).astype(np.float32)


# --------------------------------------------------
# 3. Phone matching
# --------------------------------------------------

# phone -> hospital indices
inv = defaultdict(list)

for j, ph in enumerate(h["n_phone"]):

    if isinstance(ph, str) and ph.strip():
        inv[ph.strip()].append(j)


# Store exact phone matches
hit_pairs = set()

for i, ph in enumerate(p["n_phone"]):

    if isinstance(ph, str) and ph.strip():

        ph = ph.strip()

        for j in inv.get(ph, ()):

            # Phone match gets strong score bonus
            S[i, j] += 25

            # Remember exact phone pair
            hit_pairs.add((i, j))


# --------------------------------------------------
# 4. Best candidate for each PMJAY hospital
# --------------------------------------------------

best = S.argmax(axis=1)

part = np.partition(S, -2, axis=1)

score = part[:, -1]
gap = part[:, -1] - part[:, -2]


# --------------------------------------------------
# 5. Exact phone match for BEST candidate
# --------------------------------------------------

phone_hit = np.array([
    (i, int(j)) in hit_pairs
    for i, j in enumerate(best)
])


# --------------------------------------------------
# 6. Create match dataframe
# --------------------------------------------------

m = pd.DataFrame({

    "p_idx": np.arange(len(p)),

    "h_idx": best,

    "score": score.round(1),

    "gap": gap.round(1),

    "pmjay_name": p["hospital_name"].to_numpy(),

    "h_name": h["hospital_name"].to_numpy()[best],

    "pmjay_dist": p["district"].to_numpy(),

    "h_dist": h["district"].to_numpy()[best],

    "short_name": (
        p["n_name"]
        .fillna("")
        .str.len()
        .to_numpy() < 5
    ),

    "phone_match": phone_hit,
})


# --------------------------------------------------
# 7. EXACT NORMALIZED NAME MATCH
# --------------------------------------------------

p_names = (
    p["n_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

h_names = (
    h["n_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

m["exact_name_match"] = (
    p_names.iloc[m["p_idx"]].to_numpy()
    ==
    h_names.iloc[m["h_idx"]].to_numpy()
)


# --------------------------------------------------
# 8. MATCHING RULES
#
# Priority:
#
# 1. SAME PHONE          -> AUTO
# 2. EXACT SAME NAME     -> AUTO
# 3. Strong fuzzy match  -> AUTO
# 4. Moderate match      -> REVIEW
# 5. Weak match          -> NO MATCH
# --------------------------------------------------

auto_fuzzy = (
    (m["score"] >= 90)
    & (m["gap"] >= 3)
    & ~m["short_name"]
)

m["bucket"] = np.where(

    # Rule 1: exact phone
    m["phone_match"],

    "auto",

    np.where(

        # Rule 2: exact normalized name
        m["exact_name_match"],

        "auto",

        np.where(

            # Rule 3: strong fuzzy match
            auto_fuzzy,

            "auto",

            np.where(

                # Rule 4: possible match
                m["score"] >= 75,

                "review",

                # Rule 5
                "no_match"
            )
        )
    )
)


# --------------------------------------------------
# 9. DUPLICATE DIRECTORY HOSPITAL PROTECTION
#
# Only duplicate FUZZY matches are sent to review.
#
# Phone matches and exact-name matches remain AUTO.
# --------------------------------------------------

a = m[m["bucket"] == "auto"].copy()

duplicate_auto = a.duplicated(
    "h_idx",
    keep=False
)

# Only downgrade duplicate fuzzy matches
duplicate_fuzzy = (
    duplicate_auto
    & ~a["phone_match"]
    & ~a["exact_name_match"]
)

m.loc[
    a.index[duplicate_fuzzy],
    "bucket"
] = "review"


# --------------------------------------------------
# 10. SUMMARY
# --------------------------------------------------

print("\n========== MATCH SUMMARY ==========")
print(m["bucket"].value_counts())


print("\n========== PHONE MATCHES ==========")

print(
    "Exact phone matches:",
    int(m["phone_match"].sum())
)

print(
    "Phone matches marked AUTO:",
    int(
        (
            m["phone_match"]
            & (m["bucket"] == "auto")
        ).sum()
    )
)

print(
    "Phone matches sent to REVIEW:",
    int(
        (
            m["phone_match"]
            & (m["bucket"] == "review")
        ).sum()
    )
)


print("\n========== EXACT NAME MATCHES ==========")

print(
    "Exact name matches:",
    int(m["exact_name_match"].sum())
)

print(
    "Exact name matches marked AUTO:",
    int(
        (
            m["exact_name_match"]
            & (m["bucket"] == "auto")
        ).sum()
    )
)


print("\n========== AUTO MATCH METHODS ==========")

print(
    "Phone + exact name:",
    int(
        (
            m["phone_match"]
            & m["exact_name_match"]
            & (m["bucket"] == "auto")
        ).sum()
    )
)

print(
    "Phone only:",
    int(
        (
            m["phone_match"]
            & ~m["exact_name_match"]
            & (m["bucket"] == "auto")
        ).sum()
    )
)

print(
    "Exact name only:",
    int(
        (
            m["exact_name_match"]
            & ~m["phone_match"]
            & (m["bucket"] == "auto")
        ).sum()
    )
)

print(
    "Fuzzy only:",
    int(
        (
            ~m["phone_match"]
            & ~m["exact_name_match"]
            & (m["bucket"] == "auto")
        ).sum()
    )
)


========== MATCH SUMMARY ==========
bucket
auto        1297
review       246
no_match     190
Name: count, dtype: int64

========== PHONE MATCHES ==========
Exact phone matches: 65
Phone matches marked AUTO: 65
Phone matches sent to REVIEW: 0

========== EXACT NAME MATCHES ==========
Exact name matches: 1252
Exact name matches marked AUTO: 1252

========== AUTO MATCH METHODS ==========
Phone + exact name: 65
Phone only: 0
Exact name only: 1187
Fuzzy only: 45


In [ ]:
# ---------- 5. Outputs ----------

import os

OUT_DIR = r"C:\CareNavigator\ml\data"
os.makedirs(OUT_DIR, exist_ok=True)


# ==================================================
# 1. Save REVIEW matches
# ==================================================

review = (
    m[m["bucket"] == "review"]
    .sort_values("score", ascending=False)
    .copy()
)

review.to_csv(
    os.path.join(OUT_DIR, "review_matches.csv"),
    index=False
)


# ==================================================
# 2. Save PMJAY hospitals with NO MATCH
# ==================================================

unmatched = p.loc[
    m["bucket"] == "no_match"
].copy()

unmatched.to_csv(
    os.path.join(
        OUT_DIR,
        "pmjay_unmatched_tg.csv"
    ),
    index=False
)


# ==================================================
# 3. Prepare CLEANED HOSPITAL DATA
# ==================================================

# Default values for every hospital
h["pmjay"] = "No / unverified"
h["pmjay_id"] = pd.NA
h["hospital_type"] = h.get(
    "hospital_type",
    pd.Series(pd.NA, index=h.index)
)
h["specialties"] = h.get(
    "specialties",
    pd.Series(pd.NA, index=h.index)
)
h["match_score"] = pd.NA
h["match_method"] = pd.NA


# ==================================================
# 4. Apply ALL AUTO MATCHES
# ==================================================

ok = (
    m[m["bucket"] == "auto"]
    .copy()
)

if len(ok) > 0:

    src = p.loc[
        ok["p_idx"]
    ].reset_index(drop=True)

    idx = ok["h_idx"].to_numpy()

    # PMJAY status
    h.loc[idx, "pmjay"] = "Yes"

    # PMJAY ID
    h.loc[idx, "pmjay_id"] = (
        src["hospital_id"].to_numpy()
    )

    # Hospital type
    h.loc[idx, "hospital_type"] = (
        src["hospital_type"].to_numpy()
    )

    # Specialties
    h.loc[idx, "specialties"] = (
        src["current_specialities"]
        .fillna(src["specialities_selected"])
        .to_numpy()
    )

    # Match score
    h.loc[idx, "match_score"] = (
        ok["score"]
        .round(1)
        .to_numpy()
    )

    # Match method
    h.loc[idx, "match_method"] = np.where(
        ok["phone_match"].to_numpy(),
        "Phone match",
        "Fuzzy name match"
    )


# ==================================================
# 5. Save FINAL CLEANED DATA
# ==================================================

FINAL_FILE = os.path.join(
    OUT_DIR,
    "hospitals_tg_clean.csv"
)

h.to_csv(
    FINAL_FILE,
    index=False
)
# ============================================================
# EXPAND PMJAY SPECIALTY CODES INTO FULL FORMS
# ============================================================

import pandas as pd
import os
import re

# ------------------------------------------------------------
# 1. File path
# ------------------------------------------------------------

FILE = r"C:\CareNavigator\ml\data\hospitals_tg_clean.csv"

# Load cleaned hospital dataset
df = pd.read_csv(FILE)

print("Loaded hospitals:", len(df))
print("Columns:", df.columns.tolist())


# ------------------------------------------------------------
# 2. PMJAY CODE -> FULL FORM MAPPING
# ------------------------------------------------------------

PMJAY_CODES = {

    "BM": "Burns Management",
    "CP": "Consultations & Procedures",
    "ER": "Emergency Room Packages (Care requiring less than 12 hrs stay)",

    "IA": "Name of Investigation",
    "IB": "Laboratory Medicine",
    "IC": "Nutritional Markers",
    "ID": "Infectious Diseases",
    "IE": "Hormones",
    "IF": "USG, X-Ray, CT, MRI, Bone Densitometry",
    "IG": "X-Ray",
    "IH": "X-Ray Contrast Studies",
    "II": "Mammography",
    "IJ": "CT",
    "IK": "MRI",
    "IL": "Bone Densitometry (DEXA Scan)",
    "IM": "Neurological Investigations and Procedures",
    "IMP": "Implants",
    "IN": "Interventional Neuroradiology",
    "IO": "Tests in Gastro-enterology",
    "IP": "Tests in Endocrinology (In Addition to those included under Hormones)",
    "IQ": "CSF",

    "JR": "Joint Replacement",

    "M1": "General Medicine",
    "M10": "OPD Diagnostic",
    "M2": "Paediatric Medical Management",
    "M3": "Neo-natal",
    "M4": "Paediatric Cancer",
    "M5": "Medical Oncology",
    "M6": "Radiation Oncology",
    "M7": "Emergency Room Packages (Care requiring less than 12 hrs stay)",
    "M8": "Mental Disorders Packages",

    "MA": "Treatment Procedure Skin",
    "MB": "Biopsies",
    "MC": "Cardiology",
    "MD": "Treatment Procedure ICU / CCU Procedures (Special Care Cases)",
    "ME": "Treatment Procedure Physiotherapy",
    "MF": "Nuclear Medicine / Brachytherapy - High Dose Radiation",
    "MG": "General Medicine",
    "MJ": "Nuclear Medicine / Chemotherapy",
    "MK": "List of Procedures / Tests in Gastroenterology / Endoscopic Procedures",
    "MM": "Mental Disorders Packages",
    "MN": "Neo-natal Care Packages",
    "MO": "Medical Oncology",
    "MP": "Paediatric Medical Management",
    "MQ": "Nuclear Medicine / Radiotherapy and Chemotherapy",
    "MR": "Radiation Oncology",

    "NA": "Paediatric Cancer",

    "OA": "Others",
    "OC": "OPD - Consultations, Procedures & Investigations",
    "OT": "Organ & Tissue Transplant",

    "PH": "Preventive Health Check Up",

    "S1": "General Surgery",
    "S10": "Plastic & Reconstructive",
    "S11": "Burns Management",
    "S12": "Cardiology",
    "S13": "Cardio-thoracic & Vascular Surgery",
    "S14": "Paediatric Surgery",
    "S15": "Surgical Oncology",
    "S16": "Oral and Maxillofacial Surgery",
    "S2": "Otorhinolaryngology",
    "S3": "Ophthalmology",
    "S4": "Obstetrics & Gynaecology",
    "S5": "Orthopaedics",
    "S6": "Polytrauma",
    "S7": "Urology",
    "S8": "Neurosurgery",
    "S9": "Interventional Neuroradiology",

    "SA": "Treatment Procedure Head & Neck",
    "SB": "Orthopaedics",
    "SC": "Surgical Oncology",
    "SD": "Treatment Procedure Oesophagus",
    "SE": "Ophthalmology",
    "SF": "Treatment Procedure Abdomen / GI Surgery",
    "SG": "General Surgery",
    "SH": "Treatment Procedure Burns and Plastic Surgery",
    "SI": "Treatment Procedure Nephrology and Urology",
    "SJ": "Head and Neck Cancer",
    "SK": "Treatment Procedure Breast",
    "SL": "Otorhinolaryngology",
    "SM": "Oral and Maxillofacial Surgery",
    "SN": "Neurosurgery",
    "SO": "Obstetrics & Gynaecology",
    "SP": "Plastic & Reconstructive Surgery",
    "SS": "Paediatric Surgery",
    "ST": "Polytrauma",
    "SU": "Urology",
    "SV": "Cardio-thoracic & Vascular Surgery",

    "TG": "Gender Affirming Medical and Surgical Treatments",

    "UP": "Unspecified Package"
}


# ------------------------------------------------------------
# 3. Function to expand specialty codes
# ------------------------------------------------------------

def expand_pmjay_codes(value):

    # Empty / missing value
    if pd.isna(value):
        return value

    value = str(value).strip()

    if not value:
        return value

    # --------------------------------------------------------
    # Handle different possible separators:
    #
    # MC,S5,S8
    # MC;S5;S8
    # MC | S5 | S8
    # MC / S5 / S8
    # --------------------------------------------------------

    codes = re.split(r"\s*[,;|/]\s*", value)

    expanded = []

    for code in codes:

        code = code.strip().upper()

        if not code:
            continue

        # If code exists in dictionary
        if code in PMJAY_CODES:
            expanded.append(
                f"{code} - {PMJAY_CODES[code]}"
            )

        # If value is already a full description
        else:
            expanded.append(code)

    # Remove duplicates while preserving order
    expanded = list(dict.fromkeys(expanded))

    return "; ".join(expanded)


# ------------------------------------------------------------
# 4. Expand the specialties column
# ------------------------------------------------------------

if "specialties" not in df.columns:

    raise ValueError(
        "Column 'specialties' was not found in the cleaned dataset."
    )

df["specialties"] = df["specialties"].apply(
    expand_pmjay_codes
)


# ------------------------------------------------------------
# 5. Optional: create a separate specialty_full column
# ------------------------------------------------------------
#
# This keeps the original column untouched if you want to
# compare the original data later.
#
# Currently we are replacing specialties directly, so this
# column is commented out.
#
# df["specialties_full"] = df["specialties"].apply(
#     expand_pmjay_codes
# )


# ------------------------------------------------------------
# 6. Save updated cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    FILE,
    index=False
)


# ------------------------------------------------------------
# 7. Verification
# ------------------------------------------------------------

print("\n========================================")
print("   PMJAY SPECIALTY EXPANSION COMPLETE")
print("========================================")

print("\nHospitals:", len(df))

print("\nSample expanded specialties:")

print(
    df.loc[
        df["specialties"].notna()
        & (df["specialties"].astype(str).str.strip() != ""),
        ["hospital_name", "specialties"]
    ].head(20).to_string(index=False)
)

print("\nUpdated file:")
print(FILE)

# ==================================================
# 6. Summary
# ==================================================

print("\n========================================")
print("          OUTPUT COMPLETE")
print("========================================")

print("\nOutput directory:")
print(OUT_DIR)

print("\nMatch summary:")
print(m["bucket"].value_counts())

print("\nHospital PMJAY status:")
print(h["pmjay"].value_counts())

print("\nAuto matches:", len(ok))
print(
    "  └─ Phone matches:",
    int(ok["phone_match"].sum())
)
print(
    "  └─ Fuzzy matches:",
    int((~ok["phone_match"]).sum())
)

print("\nReview matches:", len(review))
print("Unmatched PMJAY:", len(unmatched))

print("\nFinal cleaned dataset:")
print(FINAL_FILE)


Loaded cleaned hospitals: 3468

          OUTPUT COMPLETE

Output directory:
C:\CareNavigator\ml\data

Match summary:
bucket
auto        1297
review       246
no_match     190
Name: count, dtype: int64

Hospital PMJAY status:
pmjay
No / unverified    2240
Yes                1228
Name: count, dtype: int64

Auto matches: 1297
  └─ Phone matches: 65
  └─ Exact name matches: 1187
  └─ Fuzzy matches: 45

Review matches: 246
Unmatched PMJAY: 190

Final cleaned dataset:
C:\CareNavigator\ml\data\hospitals_tg_clean.csv

       SAMPLE EXPANDED SPECIALTIES
                                                   hospital_name                                                                                                                                                                                                                                                                                                                                                                                               

In [27]:
review = pd.read_csv(
    r"C:\CareNavigator\ml\data\review_matches.csv"
)

print(review.shape)
display(review)
pmjay_idx = review.iloc[0]["p_idx"]

display(
    
    review[review["p_idx"] == pmjay_idx]
    .sort_values("score", ascending=False)
)

(246, 12)


,p_idx,h_idx,score,gap,pmjay_name,h_name,pmjay_dist,h_dist,short_name,phone_match,exact_name_match,bucket
0,1015,44,106.0,0.0,Eye Care Hyderabad,Hyderabad Eye Centre,HYDERABAD,Hyderabad,False,False,False,review
1,1047,200,101.2,6.3,ARAVIND EYE HOSPITAL PRIVATE LIMITED,Arvind Eye Hospital,HYDERABAD,Hyderabad,False,False,False,review
2,1286,200,101.2,6.3,ARAVIND EYE HOSPITAL PRIVATE LIMITED,Arvind Eye Hospital,HYDERABAD,Hyderabad,False,False,False,review
3,1088,13,99.6,0.0,BASAVATARAKAM INDO AMERICAN CANCER HOSPITAL an...,Basavatarakam Indo American Cancer Institute A...,HYDERABAD,Hyderabad,False,False,False,review
4,1418,692,98.3,0.0,SAI SRINIVASA MULTISPECIALITY HOSPITAL,Sri Srinivasa Multi Speciality Hospital,NALGONDA,Nalgonda,False,False,False,review
...,...,...,...,...,...,...,...,...,...,...,...,...
241,1432,1877,75.0,0.0,AMOR HOSPITALS,Amar Hospital,MEDCHAL MALKAJGIRI,Jagitial,True,False,False,review
242,1468,1302,75.0,6.5,METROCURE HOSPITAL,Medcure Hospitals,HYDERABAD,Nagarkurnool,False,False,False,review
243,1599,1238,75.0,0.0,Roma Hospital,Rama Nursing Home,MEDCHAL MALKAJGIRI,Nalgonda,True,False,False,review
244,1615,57,75.0,0.0,BYRAM HOSPITAL,Ram Hospital,WARANGAL URBAN,Hyderabad,False,False,False,review


,p_idx,h_idx,score,gap,pmjay_name,h_name,pmjay_dist,h_dist,short_name,phone_match,exact_name_match,bucket
0,1015,44,106.0,0.0,Eye Care Hyderabad,Hyderabad Eye Centre,HYDERABAD,Hyderabad,False,False,False,review
